# HW2 - Bias in Data and Prediction

### Please complete the code or analysis under "TODO". 30pts in total. You should run every cell and keep all the outputs before submitting. Failing to include your outputs will result in zero points.
### Please keep academic integrity in mind. Plagiarism will be taken seriously.

In [15]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression

## 1. Implement Utility Functions

### 1.1 Fairness Metrics

In [ ]:
def stat_parity(preds, sens):
    '''
    :preds: numpy array of the model predictions. Consisting of 0s and 1s
    :sens: numpy array of the sensitive features. Consisting of 0s and 1s
    :return: the statistical parity. no need to take the absolute value
    '''
    p1 = preds[sens == 1].mean()
    p0 = preds[sens == 0].mean()
    return p1 - p0


def eq_oppo(preds, sens, labels):
    '''
    :preds: numpy array of the model predictions. Consisting of 0s and 1s
    :sens: numpy array of the sensitive features. Consisting of 0s and 1s
    :labels: numpy array of the ground truth labels of the outcome. Consisting of 0s and 1s
    :return: the equal opportunity difference. no need to take the absolute value
    '''

    if np.sum((sens == 1) & (labels == 1)) == 0:
        tpr1 = 0.0
    else:
        tpr1 = np.mean(preds[(sens == 1) & (labels == 1)])
    if np.sum((sens == 0) & (labels == 1)) == 0:
        tpr0 = 0.0
    else:
        tpr0 = np.mean(preds[(sens == 0) & (labels == 1)])

    return tpr1 - tpr0

In [5]:
# Test your implemented fairness metrics using the code below
# Don't change the code in this cell

# test case 1
preds = np.array([1, 0, 1, 0, 0, 1, 0, 0, 0, 1])
sens = np.array([1, 1, 0, 1, 1, 1, 0, 1, 1, 1])
labels = np.array([0, 1, 0, 1, 0, 1, 1, 1, 0, 1])
print(eq_oppo(preds, sens, labels), stat_parity(preds, sens))

# test case 2
preds = np.array([1, 1, 0, 1, 0, 1, 0, 0, 1, 1])
sens = np.array([1, 0, 0, 0, 0, 0, 0, 0, 0, 1])
labels = np.array([0, 1, 0, 1, 0, 1, 1, 0, 0, 0])
print(eq_oppo(preds, sens, labels), stat_parity(preds, sens))


# test case 3
preds = np.array([1, 0, 0, 0, 0, 0, 0, 0, 0, 1])
sens = np.array([1, 0, 0, 0, 0, 0, 0, 0, 0, 1])
labels = np.array([0, 1, 0, 1, 0, 1, 1, 0, 0, 0])
print(eq_oppo(preds, sens, labels), stat_parity(preds, sens))

0.4 -0.125
-0.75 0.5
0.0 1.0


### 1.2 Preprocessing DataFrame

In [ ]:
def process_dfs(df_train_x, df_test_x, categ_cols):
    '''
    Pre-process the features of the training set and the test set, not including the outcome column.
    Convert categorical features (nominal & ordinal features) to one-hot encodings.
    Normalize the numerical features into [0, 1].
    We process training set and the test set together in order to make sure that
    the encodings are consistent between them.
    For example, if one class is encoded as 001 and another class is encoded as 010 in the training set,
    you should follow this mapping for the test set too.

    :df_train: the dataframe of the training data
    :df_test: the dataframe of the test data
    :categ_cols: the column names of the categorical features. the rest features are treated as numerical ones.
    :return: the processed training data and test data, both should be numpy arrays, instead of DataFrames
    '''

    df_combined = pd.concat([df_train_x, df_test_x], ignore_index=True)

    # One-hot encoding categorical cols
    df_encoded = pd.get_dummies(df_combined, columns=categ_cols, dtype=float)

    # normalization
    num_cols = [col for col in df_train_x.columns if col not in categ_cols]
    for col in num_cols:
        min_val = df_encoded[col].min()
        max_val = df_encoded[col].max()
        if max_val - min_val > 0:
            df_encoded[col] = (df_encoded[col] - min_val) / (max_val - min_val)

    n_train = len(df_train_x)
    train_x = df_encoded.iloc[:n_train].values
    test_x = df_encoded.iloc[n_train:].values

    return train_x, test_x

In [10]:
# Test your implemented data preprocessing function
# DO NOT change the code in this cell

df_train_x = pd.DataFrame([
    [ 'big', 10, 'blue',],
    [ 'big', 12, 'red',],
    ['medium', 5, 'blue'],
    ['small', 7, 'yellow']
], columns=['size', 'height', 'color'])

df_test_x = pd.DataFrame([
    [ 'big', 16, 'red',],
    ['small', 9, 'blue']
], columns=['size', 'height', 'color'])

train_data_x, test_data_x = process_dfs(df_train_x, df_test_x, categ_cols=['size', 'color'])
print(train_data_x)
print()
print(test_data_x)

[[0.45454545 1.         0.         0.         1.         0.
  0.        ]
 [0.63636364 1.         0.         0.         0.         1.
  0.        ]
 [0.         0.         1.         0.         1.         0.
  0.        ]
 [0.18181818 0.         0.         1.         0.         0.
  1.        ]]

[[1.         1.         0.         0.         0.         1.
  0.        ]
 [0.36363636 0.         0.         1.         1.         0.
  0.        ]]


## 2. Load Data

In [6]:
df_train_adult = pd.read_csv('adult-train.csv', sep=', ', engine='python')
df_test_adult = pd.read_csv('adult-test.csv', sep=', ', engine='python')
df_train_adult['sex'] = df_train_adult['sex'].map({'Male': 0, 'Female': 1})
df_test_adult['sex'] = df_test_adult['sex'].map({'Male': 0, 'Female': 1})
df_train_adult['income'] = df_train_adult['income'].map({'<=50K': 0, '>50K': 1})
df_test_adult['income'] = df_test_adult['income'].map({'<=50K': 0, '>50K': 1})


df_train_german = pd.read_csv('german-train.csv')
df_test_german = pd.read_csv('german-test.csv')
df_train_german['age'] = df_train_german['age'].apply(lambda x: 1 if x >= 33 else 0)
df_test_german['age'] = df_test_german['age'].apply(lambda x: 1 if x>=33 else 0)
df_train_german['credit_status'] = df_train_german['credit_status'].map({2:0, 1:1})
df_test_german['credit_status'] = df_test_german['credit_status'].map({2:0, 1:1})

In [11]:
df_train_adult.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,0,2174,0,40,United-States,0
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,0,0,0,13,United-States,0
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,0,0,0,40,United-States,0
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,0,0,0,40,United-States,0
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,1,0,0,40,Cuba,0


In [12]:
df_train_german.head()

,checking_account,duration,credit_history,purpose,credit_amount,savings_account,present_employment_since,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,num_credits,job,num_people_liable,telephone,foreign_worker,credit_status
0,A14,21,A32,A41,5248,A65,A73,1,A93,A101,...,A123,0,A143,A152,1,A173,1,A191,A201,1
1,A11,24,A32,A43,1987,A61,A73,2,A93,A101,...,A121,0,A143,A151,1,A172,2,A191,A201,0
2,A14,36,A32,A49,5742,A62,A74,2,A93,A101,...,A123,0,A143,A152,2,A173,1,A192,A201,1
3,A14,36,A32,A49,7409,A65,A75,3,A93,A101,...,A122,1,A143,A152,2,A173,1,A191,A201,1
4,A14,6,A34,A42,1221,A65,A73,1,A94,A101,...,A122,0,A143,A152,2,A173,1,A191,A201,1


## 3. Explore fairness in data

### 3.1 statical analysis on protected feature and outcome

In [13]:
# Adult
# calculate the mean income of two protected groups. only use the training data df_train_adult.
# Protected feature: sex (0=Male, 1=Female)
mean_income1_adult = df_train_adult[df_train_adult['sex'] == 1]['income'].mean()
mean_income2_adult = df_train_adult[df_train_adult['sex'] == 0]['income'].mean()

print(mean_income1_adult, mean_income2_adult)


# German
# calculate the mean credit status of two protected groups. only use the training data df_train_german.
# Protected feature: age (0=<33, 1=33+)
mean_credit1_german = df_train_german[df_train_german['age'] == 1]['credit_status'].mean()
mean_credit2_german = df_train_german[df_train_german['age'] == 0]['credit_status'].mean()

print(mean_credit1_german, mean_credit2_german)

0.11367818442036394 0.3138370951913641
0.7594594594594595 0.6636363636363637


In [16]:
# t-test between outcome of two protected groups. only use the training data df_train_adult/german.


# Adult
income_sex1 = df_train_adult[df_train_adult['sex'] == 1]['income'].values
income_sex0 = df_train_adult[df_train_adult['sex'] == 0]['income'].values
_, p_value_adult = stats.ttest_ind(income_sex1, income_sex0)



# german
credit_age1 = df_train_german[df_train_german['age'] == 1]['credit_status'].values
credit_age0 = df_train_german[df_train_german['age'] == 0]['credit_status'].values
_, p_value_german = stats.ttest_ind(credit_age1, credit_age0)

print(p_value_adult, p_value_german)

0.0 0.005042713073567462


### From the p_values, are the results significant for Adult and German? How do you explain them?
### <span style="color:red">Please type your response here. 3pts</span>

Yes. Both p-values are less than 0.05 (Adult: p ≈ 0, German: p ≈ 0.005), so the difference in outcomes between the two protected groups is statistically significant in both datasets.

A significant p-value means the outcome (income for Adult, credit status for German) is correlated with the protected attribute (sex, age). This reflects real-world disparities in the data, e.g., the gender wage gap in income, or differences in credit outcomes by age. For fairness, this matters because a model trained on this data may learn and perpetuate these associations, leading to disparate predictions across groups.

### 3.2 Explore Fairness in Prediction

In [17]:
# Prepare data
# Dont't change code in this cell

'''
:train_x: the features in the training set (including the sensitive features), shape: N_train x d
:train_y: the outcome in the training set, shape: N_train
:test_x: the features in the test set (including the sensitive features), shape: N_test x d
:test_y: the outcome in the test set, shape: N_test
:test_sens: the sensitive/protected feature in the test set, shape: N_test
All of them are processed numpy arrays that are ready for algorithms.
'''


# adult
# the outcome (income) is the last column
df_train_x_adult = df_train_adult.iloc[:, :-1]
df_train_y_adult = df_train_adult.iloc[:, -1]
df_test_x_adult = df_test_adult.iloc[:, :-1]
df_test_y_adult = df_test_adult.iloc[:, -1]
df_test_sens_adult = df_test_adult['sex']

train_x_adult, test_x_adult = process_dfs(df_train_x_adult, df_test_x_adult,
                                                   ['workclass', 'education','marital-status',
                                                    'occupation','relationship','race',
                                                    'native-country'])
train_y_adult = df_train_y_adult.values
test_y_adult = df_test_y_adult.values
test_sens_adult = df_test_sens_adult.values

# german
# the outcome (credit status) is the last column
df_train_x_german = df_train_german.iloc[:, :-1]
df_train_y_german = df_train_german.iloc[:, -1]
df_test_x_german = df_test_german.iloc[:, :-1]
df_test_y_german = df_test_german.iloc[:, -1]
df_test_sens_german = df_test_german['age']

train_x_german, test_x_german = process_dfs(df_train_x_german, df_test_x_german,
                                                     ['checking_account', 'credit_history',
                                                      'purpose', 'savings_account', 'present_employment_since',
                                                      'personal_status_sex', 'other_debtors',
                                                     'property', 'other_installment_plans',
                                                     'housing', 'job', 'telephone', 'foreign_worker'])
train_y_german = df_train_y_german.values
test_y_german = df_test_y_german.values
test_sens_german = df_test_sens_german.values

print(train_x_adult.shape, test_x_adult.shape, train_y_adult.shape, test_y_adult.shape)
print(train_x_german.shape, test_x_german.shape, train_y_german.shape, test_y_german.shape)

(30162, 103) (15060, 103) (30162,) (15060,)
(700, 61) (300, 61) (700,) (300,)


In [18]:
# train a classifier to predict the outcome y from features x
# training: train_x --> train_y; test: test_x --> preds
# logistic regression model is recommended
# sklearn is allowed to use


# Adult

# initialize the model
model_adult = LogisticRegression(max_iter=1000, random_state=42)

# train/fit the model with train_x_adult and train_y_adult
model_adult.fit(train_x_adult, train_y_adult)


# predict the outcome from test_x_adult
preds = model_adult.predict(test_x_adult)


# report acc and two fairness metrics.
from sklearn.metrics import accuracy_score
acc = accuracy_score(test_y_adult, preds)
stat_p = stat_parity(preds, test_sens_adult)
eq_op = eq_oppo(preds, test_sens_adult, test_y_adult)
print(acc, stat_p, eq_op)





# German

# initialize the model
model_german = LogisticRegression(max_iter=1000, random_state=42)


# train/fit the model with train_x_german and train_y_german
model_german.fit(train_x_german, train_y_german)


# predict the outcome from test_x_german
preds = model_german.predict(test_x_german)


# report acc and two fairness metrics
from sklearn.metrics import accuracy_score
acc = accuracy_score(test_y_german, preds)
stat_p = stat_parity(preds, test_sens_german)
eq_op = eq_oppo(preds, test_sens_german, test_y_german)
print(acc, stat_p, eq_op)

0.846215139442231 -0.1836180144547651 -0.1016433315378108
0.7566666666666667 0.10337468320661602 0.08974358974358976
